# Legal-model extension: comparison of all arms

Compares the {Llama-3.1-8B, Mistral-7B, Saul-7B-Base} × {zero-shot, QLoRA} grid on the three tasks
(T1 CUAD risk clauses, T2 CUAD entity extraction, T3 LEDGAR provisions). Design, run log and caveats:
[docs/extension/DESIGN.md](docs/extension/DESIGN.md).

**Inputs** (CPU only, nothing is trained here):
- `kaggle_output_extension/extension_arms.csv` and `extension_comparisons.csv`, written by
  `python scripts/collect_extension_results.py` (re-run it after downloading new runs);
- each harness run's `predictions.jsonl` / `eval_metrics.json` under `kaggle_output_extension/runs/`,
  for per-arm confidence intervals and per-category / per-label breakdowns.

**Metrics** (same as `legal_model_extension.py compare`): *correctness* = T1 accuracy / T2 exact match /
T3 accuracy; *headline F1* = T1 macro-F1 / T2 mean value F1 / T3 macro-F1 over 100 labels; *validity* =
T1 strict Yes/No / T2 strict JSON / T3 valid label. "Llama QLoRA" is the original adapter re-scored
through the harness (fp16); "Llama zero-shot" comes from the original bf16 notebooks and has no
per-example predictions, so it has no CI and is in no paired test.

**Caveat:** Saul's pretraining corpus includes SEC EDGAR, the source of both CUAD and LEDGAR.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(".")
RESULTS = ROOT / "kaggle_output_extension"
RUNS = RESULTS / "runs"
sys.path.insert(0, str(ROOT / "scripts"))
import task1_metrics, task2_metrics, task3_metrics   # the harness's own scorers

task3_metrics.load_labels(ROOT / "data" / "LEDGAR" / "labels.json")
METRICS = {1: task1_metrics, 2: task2_metrics, 3: task3_metrics}
TASK_NAME = {1: "T1 risk clauses", 2: "T2 entity extraction", 3: "T3 LEDGAR provisions"}
METRIC_NAMES = {1: ("strict Yes/No", "accuracy", "macro-F1"),
                2: ("strict JSON", "exact match", "mean F1"),
                3: ("valid label", "accuracy", "macro-F1")}
MODELS = ["llama", "mistral", "saul"]
MODEL_LABEL = {"llama": "Llama-3.1-8B", "mistral": "Mistral-7B", "saul": "Saul-7B"}
# Categorical slots 1-3 of the validated default palette, fixed per model (never by rank).
MODEL_COLOR = {"llama": "#2a78d6", "mistral": "#eb6834", "saul": "#1baf7a"}
FINETUNED = {(t, m): r for t in (1, 2, 3) for m, r in
             {"llama": f"llama_t{t}_adapter_adapter{t}", "mistral": f"mistral_t{t}_finetune",
              "saul": f"saul_t{t}_finetune"}.items()}
N_BOOT, SEED = 2000, 0   # same as every `compare` run

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.6,
                     "axes.edgecolor": "#8a8984", "axes.labelcolor": "#52514e",
                     "xtick.color": "#52514e", "ytick.color": "#52514e"})

arms = pd.read_csv(RESULTS / "extension_arms.csv")
comparisons = pd.read_csv(RESULTS / "extension_comparisons.csv")
print(f"{len(arms)} arms, {len(comparisons)} paired comparisons")

In [ ]:
def load_predictions(run):
    with open(RUNS / run / "predictions.jsonl", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def bootstrap_ci(task, preds):
    # Percentile 95% CI of the task's headline F1, resampling examples (seeded like `compare`).
    gold = [p["gold"] for p in preds]
    pred = [p["pred"] for p in preds]
    rng = np.random.default_rng(SEED)
    stats = []
    for _ in range(N_BOOT):
        idx = rng.integers(0, len(gold), len(gold))
        stats.append(METRICS[task].headline_f1([gold[i] for i in idx], [pred[i] for i in idx]))
    return np.percentile(stats, [2.5, 97.5])


PREDS = {run: load_predictions(run) for run in arms["run"] if (RUNS / run / "predictions.jsonl").exists()}
ci = {run: bootstrap_ci(int(arms.loc[arms.run == run, "task"].iloc[0]), p) for run, p in PREDS.items()}
arms["f1_ci_low"] = arms["run"].map(lambda r: ci[r][0] if r in ci else np.nan)
arms["f1_ci_high"] = arms["run"].map(lambda r: ci[r][1] if r in ci else np.nan)
print(f"Bootstrapped {len(ci)} harness arms ({N_BOOT} resamples each)")

## 1. Headline table per task (all six arms)

In [ ]:
def fmt_ci(row):
    return "" if np.isnan(row.f1_ci_low) else f"[{row.f1_ci_low:.3f}, {row.f1_ci_high:.3f}]"


for task in (1, 2, 3):
    validity, correct, f1 = METRIC_NAMES[task]
    t = arms[arms.task == task].copy()
    t["model"] = t["model"].map(MODEL_LABEL)
    t["95% CI"] = t.apply(fmt_ci, axis=1)
    t = t.rename(columns={"validity": validity, "correctness": correct, "headline_f1": f1})
    print(f"\n{TASK_NAME[task]} (n = {t.n.iloc[0]:,})")
    display(t[["arm", "model", validity, correct, f1, "95% CI", "source"]]
            .sort_values(["arm", "model"]).round(3).reset_index(drop=True))

## 2. The three fine-tuned models

Llama-3.1-8B (original QLoRA adapter, re-scored), Mistral-7B QLoRA and Saul-7B QLoRA, each trained once
with identical data, prompts, hyperparameters and decoding. Mistral vs Saul isolates legal pretraining
(same architecture and tokenizer); either vs Llama compares base models.

### 2a. Headline metrics with 95% bootstrap CIs

In [ ]:
ft = arms[arms.arm == "qlora"].copy()
table = ft.assign(model=ft.model.map(MODEL_LABEL), ci=ft.apply(fmt_ci, axis=1))[
    ["task", "model", "validity", "correctness", "headline_f1", "ci", "train_hours"]]
display(table.rename(columns={"headline_f1": "headline F1", "ci": "95% CI",
                              "train_hours": "train h"}).round(3).reset_index(drop=True))

# Dot + interval per model, one panel per task (own scale: the gaps are a few thousandths).
fig, axes = plt.subplots(1, 3, figsize=(11, 2.8))
for ax, task in zip(axes, (1, 2, 3)):
    t = ft[ft.task == task].set_index("model").loc[MODELS]
    for y, (model, row) in enumerate(t.iterrows()):
        ax.plot([row.f1_ci_low, row.f1_ci_high], [y, y], color=MODEL_COLOR[model], lw=2,
                solid_capstyle="round")
        ax.plot(row.headline_f1, y, "o", ms=8, color=MODEL_COLOR[model], mec="#fcfcfb", mew=2)
        ax.annotate(f"{row.headline_f1:.3f}", (row.f1_ci_high, y), xytext=(6, 0),
                    textcoords="offset points", va="center", fontsize=8, color="#52514e")
    ax.set_yticks(range(3), [MODEL_LABEL[m] for m in MODELS] if task == 1 else [])
    ax.set_ylim(-0.6, 2.6)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_title(f"{TASK_NAME[task]}\n{METRIC_NAMES[task][2]}", fontsize=9, loc="left")
    lo, hi = t.f1_ci_low.min(), t.f1_ci_high.max()
    ax.set_xlim(lo - 0.01, hi + 0.02)
fig.suptitle("Fine-tuned models: headline F1 with 95% bootstrap CI", x=0.01, ha="left", fontsize=11)
fig.tight_layout()
plt.show()

### 2b. Pairwise paired tests

McNemar's exact test on per-example correctness and the paired-bootstrap 95% CI of the headline-F1
difference (B − A). The three pairs are tested on the same data, so read a single p near 0.01–0.05 as
weak evidence (Bonferroni for three pairs: 0.0167).

In [ ]:
def finetuned_model(run):
    return next((m for (t, m), r in FINETUNED.items() if r == run), None)


pairs = comparisons.assign(model_a=comparisons.a.map(finetuned_model),
                           model_b=comparisons.b.map(finetuned_model)).dropna(subset=["model_a", "model_b"])
pairs = pairs.assign(
    pair=pairs.model_a.map(MODEL_LABEL) + " → " + pairs.model_b.map(MODEL_LABEL),
    discordant=pairs.only_a_correct.astype(str) + " / " + pairs.only_b_correct.astype(str),
    f1_diff=pairs.apply(lambda r: f"{r.f1_diff_b_minus_a:+.3f} [{r.f1_diff_ci_low:+.3f}, {r.f1_diff_ci_high:+.3f}]", axis=1),
    significant=(pairs.mcnemar_p < 0.05 / 3) | (pairs.f1_diff_ci_low > 0) | (pairs.f1_diff_ci_high < 0))
display(pairs[["task", "pair", "discordant", "mcnemar_p", "f1_diff", "significant"]]
        .rename(columns={"discordant": "only A / only B correct", "mcnemar_p": "McNemar p",
                         "f1_diff": "F1 diff B−A [95% CI]", "significant": "p<0.0167 or CI excludes 0"})
        .sort_values(["task", "pair"]).round(4).reset_index(drop=True))

### 2c. T1 per category (accuracy)

In [ ]:
def per_group_accuracy(task, key="category"):
    frames = []
    for model in MODELS:
        preds = PREDS[FINETUNED[(task, model)]]
        df = pd.DataFrame({key: [p[key] for p in preds], model: [p["correct"] for p in preds]})
        frames.append(df.groupby(key)[model].mean())
    counts = pd.Series([p[key] for p in PREDS[FINETUNED[(task, "saul")]]]).value_counts().rename("n")
    out = pd.concat(frames + [counts], axis=1).rename(columns=MODEL_LABEL)
    out["spread"] = out[[MODEL_LABEL[m] for m in MODELS]].max(axis=1) - out[[MODEL_LABEL[m] for m in MODELS]].min(axis=1)
    return out


t1_categories = per_group_accuracy(1)
display(t1_categories.sort_values("spread", ascending=False).round(3))

### 2d. T2 per category (exact match), and the overall summary without `Warranty Duration`

`Warranty Duration` has only 11 validation examples, so it is shown per category but excluded from
the second overall row. Agreement/Effective Date golds are punctuation-stripped digit strings
(`5/8/14` → `5814`), which favours the digit-by-digit Mistral/Saul tokenizer — see DESIGN.md.

In [ ]:
t2_categories = per_group_accuracy(2)
display(t2_categories.round(3))


def t2_overall(model, exclude=()):
    preds = [p for p in PREDS[FINETUNED[(2, model)]] if p["category"] not in exclude]
    return {"n": len(preds), "exact match": np.mean([p["correct"] for p in preds]),
            "mean F1": task2_metrics.headline_f1([p["gold"] for p in preds], [p["pred"] for p in preds])}


rows = []
for label, exclude in [("all categories", ()), ("without Warranty Duration", ("Warranty Duration",)),
                       ("without Warranty Duration and the two date categories",
                        ("Warranty Duration", "Agreement Date", "Effective Date"))]:
    for model in MODELS:
        rows.append({"subset": label, "model": MODEL_LABEL[model], **t2_overall(model, exclude)})
display(pd.DataFrame(rows).pivot(index="subset", columns="model", values=["exact match", "mean F1"]).round(3))

### 2e. T3 per label (F1 over 100 labels)

Per-label F1 from each run's `eval_metrics.json`. The scatter is the plan's Saul-vs-Mistral comparison
(legal pretraining at label level); points above the diagonal are labels where Saul is better.

In [ ]:
per_label = pd.DataFrame({
    MODEL_LABEL[m]: {label: v["f1"] for label, v in json.loads(
        (RUNS / FINETUNED[(3, m)] / "eval_metrics.json").read_text(encoding="utf-8"))["per_label"].items()}
    for m in MODELS})
saul, mistral = MODEL_LABEL["saul"], MODEL_LABEL["mistral"]
per_label["saul − mistral"] = per_label[saul] - per_label[mistral]
print(f"Labels where Saul > Mistral: {(per_label['saul − mistral'] > 0).sum()}, "
      f"Mistral > Saul: {(per_label['saul − mistral'] < 0).sum()}, equal: {(per_label['saul − mistral'] == 0).sum()}")
print("Largest per-label differences (n = 20 per label, so a 0.1 F1 gap is about two examples):")
display(per_label.reindex(per_label["saul − mistral"].abs().sort_values(ascending=False).index).head(10).round(3))

fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.plot([0, 1], [0, 1], color="#8a8984", lw=1, ls="--", zorder=1)
ax.scatter(per_label[mistral], per_label[saul], s=36, color=MODEL_COLOR["saul"], edgecolor="#fcfcfb",
           linewidth=1.5, zorder=2)
for label, row in per_label.reindex(per_label["saul − mistral"].abs().sort_values(ascending=False).index).head(4).iterrows():
    ax.annotate(label, (row[mistral], row[saul]), xytext=(5, -3), textcoords="offset points", fontsize=7,
                color="#52514e")
ax.set_xlabel(f"{mistral} per-label F1")
ax.set_ylabel(f"{saul} per-label F1")
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.set_title("T3 per-label F1, Saul vs Mistral (QLoRA)\nabove the diagonal = Saul better", fontsize=10, loc="left")
fig.tight_layout()
plt.show()

## 3. Validity versus content (T2 and T3)

Separates *format* (does the model produce an answer the scorer can read) from *content* (is a readable
answer right). Content here = correctness among valid answers. The three QLoRA points overlap at the
top right of each panel (see the table). Zero-shot gaps between models are almost
entirely format; fine-tuning brings every model to ≈ 0.995 validity, after which only content differs.

In [ ]:
from matplotlib.lines import Line2D

rows = []
for task in (2, 3):
    for _, arm in arms[arms.task == task].iterrows():
        if arm.run in PREDS:
            preds = PREDS[arm.run]
            invalid = task2_metrics.INVALID if task == 2 else task3_metrics.INVALID
            valid = [p for p in preds if p["pred"] != invalid]
            validity = len(valid) / len(preds)
            content = np.mean([p["correct"] for p in valid]) if valid else np.nan
        else:
            # Llama zero-shot (notebook, aggregates only): an invalid answer is never correct,
            # so correct-among-valid = correctness / validity exactly.
            validity, content = arm.validity, arm.correctness / arm.validity
        rows.append({"task": task, "arm": arm.arm, "model": arm.model, "validity": validity,
                     "correct | valid": content, "source": arm.source})
vc = pd.DataFrame(rows)
display(vc.assign(model=vc.model.map(MODEL_LABEL)).sort_values(["task", "arm", "model"]).round(3)
        .reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, task in zip(axes, (2, 3)):
    for _, r in vc[vc.task == task].iterrows():
        filled = r.arm == "qlora"
        ax.scatter(r.validity, np.nan_to_num(r["correct | valid"]), s=70, zorder=3, linewidth=2,
                   color=MODEL_COLOR[r.model] if filled else "#fcfcfb", edgecolor=MODEL_COLOR[r.model])
    ax.set_xlim(-0.03, 1.05)
    ax.set_ylim(-0.03, 1.03)
    ax.set_xlabel(f"validity ({METRIC_NAMES[task][0]})")
    ax.set_title(TASK_NAME[task], fontsize=10, loc="left")
axes[0].set_ylabel("correct among valid answers")
handles = [Line2D([], [], marker="o", ls="", ms=8, color=MODEL_COLOR[m], label=MODEL_LABEL[m]) for m in MODELS]
handles += [Line2D([], [], marker="o", ls="", ms=8, mfc="#fcfcfb", mec="#52514e", mew=2, label="zero-shot (hollow)"),
            Line2D([], [], marker="o", ls="", ms=8, color="#52514e", label="QLoRA (filled)")]
fig.legend(handles=handles, loc="upper right", ncol=5, fontsize=8, frameon=False)
fig.suptitle("Format (x) versus content (y)", x=0.01, ha="left", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

## 4. T1: lenient accuracy versus strict validity

T1 accuracy uses the notebooks' lenient parse (anything not starting with "Yes" counts as "No"), so a
model that rambles is silently scored as answering "No". The strict-validity rate (first line exactly
Yes/No) shows how much of each arm's accuracy rests on that rule. Saul zero-shot's 0.747 is its
all-"No" answer matching the 75 % "No" majority, not skill.

In [ ]:
t1 = arms[arms.task == 1].copy()
t1["yes predicted"] = t1.run.map(lambda r: np.mean([p["pred"] == "Yes" for p in PREDS[r]]) if r in PREDS else np.nan)
display(t1.assign(model=t1.model.map(MODEL_LABEL))[
    ["arm", "model", "correctness", "validity", "headline_f1", "secondary", "yes predicted", "source"]]
    .rename(columns={"correctness": "lenient accuracy", "validity": "strict validity",
                     "headline_f1": "macro-F1", "secondary": "Yes-F1"})
    .sort_values(["arm", "model"]).round(3).reset_index(drop=True))
print("Gold 'Yes' rate:", round(np.mean([p["gold"] == "Yes" for p in PREDS["saul_t1_finetune"]]), 3))

## 5. Summary

- **Fine-tuning is what matters.** Every model gains +0.38 to +0.85 headline F1 from QLoRA; zero-shot,
  none is usable on every task (Saul answers "No" to all of T1; Mistral fences and never stops on T2 and
  echoes the label list on T3).
- **Legal pretraining (Saul vs Mistral, same architecture and tokenizer) shows no measurable gain after
  fine-tuning on any task** (T1 p = 0.20, T2 p = 0.52, T3 p = 0.78; every F1 CI includes 0).
- **The three fine-tuned models are close.** Mistral leads Llama on T1 (p = 0.007); Saul and Mistral
  lead Llama on T2 only through the digit-string date labels (a tokenizer effect); T3 is a three-way tie.
- Caveats: one training seed per arm; Llama adapters were trained in bf16 by the original notebooks;
  Saul's pretraining includes EDGAR (contamination).